# VQ2 training on Colab

Trains the 3-DoF racing policy against the surrogate, on the **measured VQ2 course**
(`pilot/control/course/`, 17 gates) rather than the procedural generator.

Run the cells top to bottom. Cell 4 is the one you re-run with different knobs.

**Use a CPU runtime.** The environment is vectorized NumPy on CPU and dominates the step;
the network is a 2x256 MLP, so a GPU only accelerates the PPO update and does not change
the picture much. The real win here is **running several of these notebooks at once** with
different settings -- the bottleneck is hypotheses per hour, not steps per second.

No `pip install` step: `pilot/control/` needs only numpy + torch, both preinstalled.
It does **not** import cv2, so the `numpy==1.26.4` / `opencv-python==4.10.0.84` pin in
CLAUDE.md does not apply here -- the tree runs fine on Colab's numpy 2.x.

## 1. Get the code

`BRANCH` must exist **on origin**. A local-only branch cannot be cloned; push it first.

In [ ]:
REPO   = "clarity-m/vqual-2"              #@param {type:"string"}
BRANCH = "worktree-vq2-reward-eval"       #@param {type:"string"}

import os, subprocess, getpass

DEST = "/content/vqual-2"
if not os.path.exists(DEST):
    tok = getpass.getpass("GitHub token (blank if the repo is public): ").strip()
    url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "checkout", "-f", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(DEST)
print("cwd:", os.getcwd())
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True,
                              text=True).stdout.strip())
print("course package present:", os.path.exists("pilot/control/course/course_vq2.json"))

## 2. Check the environment and verify the harness

All three should pass before you spend hours on a run. The third is the train/deploy
agreement test: it drives six {consecutive, dilated} x {affine, residual} x {plain,
vertical-rate} combinations through both the training path and `RLPolicy` and requires
identical physical commands. If it fails, a checkpoint that scores well will fly
differently on deployment -- and both would look entirely normal.

In [ ]:
import sys, torch, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), "| numpy", numpy.__version__)

!python pilot/control/train/selftest.py 2>&1 | tail -3
!python pilot/control/surrogate/selfcheck.py 2>&1 | tail -2
!python pilot/control/train/tests/test_train_deploy_agreement.py 2>&1 | tail -8

## 3. Persist checkpoints to Drive (optional but recommended)

Colab reclaims the VM without warning. `train.py` always writes to
`pilot/control/train/checkpoints/`, so pointing that at Drive is enough -- no code change,
and `--resume` then works across a disconnect.

In [ ]:
USE_DRIVE = True                                        #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/vqual2-checkpoints" #@param {type:"string"}

import os, shutil
CKPT = "pilot/control/train/checkpoints"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    if os.path.islink(CKPT):
        os.unlink(CKPT)
    elif os.path.isdir(CKPT):
        shutil.rmtree(CKPT)
    os.symlink(DRIVE_DIR, CKPT)
    print("checkpoints ->", os.path.realpath(CKPT))
else:
    os.makedirs(CKPT, exist_ok=True)
    print("checkpoints stay on the VM and are LOST on disconnect:", CKPT)

## 4. Train

### The three architecture flags

All default OFF; a run with all three off is bit-identical to the old behaviour. They are
measured on **observability**, not on task performance -- none has been trained to
convergence, which is exactly what this notebook is for.

| flag | what it does | measured |
|---|---|---|
| `VERTICAL_RATE` | appends 4 derived channels: leaky integrals of world-frame vertical acceleration | R2 0.630 vs 0.460 predicting true v_z |
| `THRUST_RESIDUAL` | `u[2]=0` means "hold altitude" at any attitude, instead of the net learning `1/(cos roll cos pitch)` | corrects a term 0.24-0.43x the policy's own exploration sigma |
| `FRAME_OFFSETS` | same 6 frames spread over 0.58 s instead of 0.111 s; identical width and parameter count | R2 0.460 -> 0.562 |

The first two target the floor-strike failure that dominated run1 (55.5% of episodes).
`FRAME_OFFSETS` is the weakest of the three and is a free add; it is not the fix.

### The course

`VQ2_FRAC` is the share of episodes flown on the measured course. `1.0` trains on VQ2
only, which is the right call when the goal is to optimize for this specific course --
it is fixed and deterministic (spec 3.5), attempts are unlimited and ranking is on time
(9.4), so overfitting to it is the point.

The one hedge worth considering is `0.9`. `course/README.md` warns that a whole leg can
rotate by 90 deg if its mod-90 quadrant came from the sketch rather than a shared
measurement -- "a discrete failure a Gaussian cannot express" -- and calls the sampled
envelope "a lower bound". A pure-VQ2 policy has memorized a track that may be wrong in a
way no amount of sampling covers; a 10% procedural share keeps gate-seeking alive for that
case. Your call.

`TOTAL_STEPS` is an **absolute** target, not steps-this-session. Set `RESUME` to a
checkpoint name to continue a run instead of starting fresh -- but note that resuming
without a `<stem>_resume.pt` sidecar is a degraded continuation (weights only, no
Adam/curriculum/RNG state) and prints a banner saying so.

In [ ]:
NAME             = "vq2_arch1"     #@param {type:"string"}
TOTAL_STEPS      = 200000000       #@param {type:"integer"}
VQ2_FRAC         = 1.0             #@param {type:"slider", min:0, max:1, step:0.05}
N_ENVS           = 2048            #@param {type:"integer"}
N_STEPS          = 128             #@param {type:"integer"}
FRAME_STACK      = 6               #@param {type:"integer"}
GATES_PER_EP     = 22              #@param {type:"integer"}
CURRIC_WINDOW    = 1000            #@param {type:"integer"}
ENT_COEF         = 0.001           #@param {type:"number"}
SEED             = 1               #@param {type:"integer"}
DEVICE           = "cpu"           #@param ["cpu", "cuda"]

VERTICAL_RATE    = True            #@param {type:"boolean"}
THRUST_RESIDUAL  = True            #@param {type:"boolean"}
FRAME_OFFSETS    = ""              #@param {type:"string"}

RESUME           = ""              #@param {type:"string"}
EXTRA            = ""              #@param {type:"string"}

cmd = [
    "python", "pilot/control/train/train.py",
    "--env", "surrogate",
    "--name", NAME,
    "--total-steps", str(TOTAL_STEPS),
    "--n-envs", str(N_ENVS),
    "--n-steps", str(N_STEPS),
    "--frame-stack", str(FRAME_STACK),
    "--gates-per-episode", str(GATES_PER_EP),
    "--curriculum-window", str(CURRIC_WINDOW),
    "--ent-coef", str(ENT_COEF),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--checkpoint-every", "1000000",
    "--env-kwarg", f"vq2_frac={VQ2_FRAC}",
]
if VERTICAL_RATE:
    cmd += ["--vertical-rate"]
if THRUST_RESIDUAL:
    cmd += ["--thrust-residual"]
if FRAME_OFFSETS.strip():
    cmd += ["--frame-offsets", FRAME_OFFSETS.strip()]
if RESUME:
    cmd += ["--resume", RESUME]
if EXTRA:
    cmd += EXTRA.split()

print(" ".join(cmd), "\n")

import subprocess
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print("\nexit code:", p.returncode)

## 5. Reading the log

```
[ 12/610] step=393216 fps=9500 ret=-2.9 gates=1.91 coll=0.995 compl=0.00 | diff=0.00 speed_cap=0.50 ...
```

| field | meaning |
|---|---|
| `compl` | completion rate -- the number that matters. Curriculum promotes above 0.70 |
| `coll`  | collision rate. Starts near 1.0 and should fall |
| `gates` | mean gates passed before the episode ends (17 is a full VQ2 lap) |
| `ev`    | value-function explained variance. Should climb early; if it stays near 0 something is wrong |
| `diff` / `speed_cap` | the two curriculum axes, moved automatically |

Starting at `compl=0.00` with `coll` near 1.0 is expected, not a failure. The thing to
watch for is run1's outcome: ~5M steps at `compl=0.00` **throughout**, so the curriculum
never promoted. If `compl` is still flat at 0 by a few M steps, kill it rather than let it
burn the VM.

Per-update rows also land in `checkpoints/<NAME>_log.csv`.

## Running several at once

Duplicate this notebook and change `NAME` and `SEED` (and whatever you are testing).
Distinct `NAME`s never collide -- checkpoints, sidecars and logs are all name-keyed.
The comparisons worth spending parallel VMs on:

- **the ablation that matters**: `VERTICAL_RATE`+`THRUST_RESIDUAL` on vs both off, same
  seed. Nothing else tells you whether the flags earned their place
- `FRAME_OFFSETS = "32,16,8,4,2,0"` -- dilated stack, 0.58 s span, same parameter count
- `VQ2_FRAC` -- 0.0 (pure procedural, the control) vs 0.9 vs 1.0
- `EXTRA = "--env-kwarg vq2_yaw_mode='bisector'"` -- fixed gate planes instead of
  randomizing across the three disagreeing yaw hypotheses
- `EXTRA = "--difficulty-start 0.3 --speed-cap-start 0.7"` -- start the curriculum higher
- `ENT_COEF` -- 0.005 -> 0.001 bought real accuracy and the lever is not exhausted. Note
  `log_std` is state-independent, so this only scales a fixed isotropic exploration blob;
  the entropy bonus has zero gradient into either trunk

## Ranking checkpoints -- match the course

The eval suite defaults to PROCEDURAL courses. Ranking a VQ2-trained checkpoint against
courses it never saw selects the wrong one, silently. Always pass `--vq2-frac` matching
what you trained on:

```
!python pilot/control/evalsuite/select.py --ckpt vq2_arch1_s5046272 \
    --ckpt vq2_arch1_s10092544 --baseline --vq2-frac 1.0 --seeds 0-49
```

Checkpoints self-describe: `frame_offsets`, `thrust_residual` and `vertical_rate` are
written into the checkpoint only when in use, so `RLPolicy` reconstructs the right
architecture with no flags at eval or deploy time.

## Known gaps

- Gate 9's tilt is held at vertical pending the corrected `course_vq2.json`. Once it
  lands, add `--env-kwarg "vq2_tilt_deg={9:(21.0,24.0)}"` via `EXTRA`.
- The **noise model is optimistic in three measured ways** (`perception-error.md`):
  rho(0.1 s) 0.954 modelled vs 0.43 measured; `p_detect` 0.62-0.92 modelled vs 0.53-0.60
  measured with 0.57 s p90 blackouts; and the size-fallback range bias is 3.5 m **short**,
  not long. Anything trained here has never seen the dropouts it will fly through live.